# NovoBench Metrics

In [1]:
import sys
sys.path.append("..")
from pathlib import Path
from dataclasses import dataclass, fields
from concurrent.futures import ProcessPoolExecutor, Future

import pandas as pd

from rocnovo.metrcis.abnovobench import aa_match_metrics, aa_match_batch
from rocnovo.tokenizer.peptide import PTMPeptideTokenizer

DEFAULT_DATASETS = ["v1", "v2"]

@dataclass
class Metric:
    aa_precision: float
    aa_recall: float
    curve_auc: float
    pep_precision: float
    pep_recall: float
    ptm_precision: float
    ptm_recall: float
    full_accuracy: float

def calculate_metrics(df: pd.DataFrame, name: str):
    ptm_list = [
        "C+57.021",
        "M+15.995",
        "N+0.984",
        "Q+0.984"
    ]
    if name == "hc_pt":
        ptm_list.remove("C+57.021")
    
    tokenizer = PTMPeptideTokenizer(
        residues="massivekb",
        reverse=False
    )
    batch, n_pep, n_aa_true, n_aa_pred, n_ptm_true, n_ptm_pred = aa_match_batch(
        df["gt_peptide"].to_list(),
        df["pred_peptide"].to_list(),
        tokenizer.masses,
        ptm_list,
        0.5,
        0.1,
        "best"
    )
    metrics = aa_match_metrics(
        batch,
        n_pep,
        n_aa_true,
        n_aa_pred,
        n_ptm_true,
        n_ptm_pred,
        df["pred_score"]
    )
    return Metric(
        metrics["aa_precision"],
        metrics["aa_recall"],
        metrics["curve_auc"],
        metrics["pep_precision"],
        metrics["pep_recall"],
        metrics["ptm_precision"],
        metrics["ptm_recall"],
        (df["pred_peptide"] == df["gt_peptide"]).mean()
    )

def pipeline(
    root_dir: Path,
    target_datasets: list[str] = None,
    prefix: str = "",
    path_suffix: str=".csv"
) -> pd.DataFrame:
    raw_names = target_datasets if target_datasets is not None else DEFAULT_DATASETS
    folder_names = [f"{prefix}_{name}" if prefix else name for name in raw_names]
    
    dataset_item_sets: list[set[str]] = []
    valid_folders: list[str] = []
    
    for folder in folder_names:
        folder_path = root_dir / folder
        if folder_path.exists() and folder_path.is_dir():
            items = {p.stem for p in folder_path.iterdir() if p.suffix == path_suffix}
            dataset_item_sets.append(items)
            valid_folders.append(folder)
    
    if not dataset_item_sets:
        return pd.DataFrame()
    
    common_items = set.intersection(*dataset_item_sets)
    if not common_items:
        return pd.DataFrame()
    
    species_results = {item: {"species": item} for item in common_items}
    future_to_meta: dict[Future, dict[str, str]] = {}
    with ProcessPoolExecutor(max_workers=30) as executor:
        for item in sorted(common_items):
            for folder in valid_folders:
                subset_result_path = root_dir / folder / f"{item}{path_suffix}"

                if not subset_result_path.exists():
                    continue
                
                if path_suffix == ".csv":
                    df = pd.read_csv(
                        subset_result_path,
                        sep=",",
                        keep_default_na=False
                    )
                
                future = executor.submit(calculate_metrics, df, item)
                future_to_meta[future] = {
                    "item": item,
                    "folder": folder
                }
        
        for future, meta in future_to_meta.items():
            metric = future.result()
            item = meta["item"]
            folder = meta["folder"]
            
            for f in fields(metric):
                species_results[item][f"{folder}_{f.name}"] = getattr(metric, f.name)
    
    records = [species_results[item] for item in sorted(common_items)]
    df = pd.DataFrame(records)
    if not df.empty:
        df.loc[len(df)] = ["mean", *df.mean(numeric_only=True).values]
    
    return df

In [2]:
from rocnovo.common.io import normalize_path

novobench_df = pipeline(
    normalize_path("/data2/xp/RocNovo-Lightning/outputs/final"),
    ["novobench_results"]
)
novobench_df

100%|██████████| 28572/28572 [00:01<00:00, 19623.73it/s]


,species,novobench_results_aa_precision,novobench_results_aa_recall,novobench_results_curve_auc,novobench_results_pep_precision,novobench_results_pep_recall,novobench_results_ptm_precision,novobench_results_ptm_recall,novobench_results_full_accuracy
0,hc_pt,0.670393,0.670120,0.471608,0.512909,0.512909,0.742892,0.779689,0.511821
1,nine_species,0.832253,0.832956,0.632845,0.664847,0.664847,0.834906,0.765034,0.654137
2,seven_species,0.568318,0.570527,0.309827,0.367587,0.367587,0.543753,0.513859,0.356877
3,mean,0.690321,0.691201,0.471426,0.515114,0.515114,0.707184,0.686194,0.507612


# MassiveKB Zero-Shot Metrics

The model weights for this MassIVE-KB version were trained using realistic data augmentation.

In [3]:
from pathlib import Path
from dataclasses import dataclass, fields
from concurrent.futures import Future, ProcessPoolExecutor

import pandas as pd

from rocnovo.common.io import normalize_path
from rocnovo.tokenizer.peptide import PTMPeptideTokenizer
from rocnovo.metrcis.accuracy import aa_match_batch, aa_match_metrics

DEFAULT_DATASETS = ["v1", "v2"]

@dataclass
class Metric:
    aa_precision: float
    aa_recall: float
    peptide_recall: float
    full_accuracy: float

def calculate_metrics(df: pd.DataFrame):
    tokenizer = PTMPeptideTokenizer(
        residues="massivekb",
        reverse=False
    )
    aa_matches_batch, n_aa1, n_aa2 = aa_match_batch(
        df["gt_peptide"].tolist(),
        df["pred_peptide"].tolist(),
        tokenizer.masses,
        mode="best"
    )
    aa_precision, aa_recall, pep_recall = aa_match_metrics(
        aa_matches_batch,
        n_aa1,
        n_aa2
    )
    return Metric(
        aa_precision,
        aa_recall,
        pep_recall,
        (df["pred_peptide"] == df["gt_peptide"]).mean()
    )

def calculate_multi_strategy_metrics(dir_path: Path, version: str="v1"):
    records: list[dict[str, float]] = []
    future_to_meta: dict[Future, dict[str, str]] = {}
    
    with ProcessPoolExecutor(max_workers=50) as executor:
        for sub_dir in sorted(dir_path.iterdir()):
            if not sub_dir.is_dir():
                continue
            
            target_dir = sub_dir / version
            if not target_dir.exists():
                continue
            
            for path in sorted(target_dir.iterdir()):
                if path.suffix != '.csv': 
                    continue
                
                df = pd.read_csv(
                    path,
                    sep=",",
                    keep_default_na=False
                )
                
                future = executor.submit(calculate_metrics, df)
                
                future_to_meta[future] = {
                    "Search Strategy": sub_dir.stem,
                    "Species": path.stem
                }
        
        for future, meta in future_to_meta.items():
            metric = future.result()
            item = meta.copy()
            for f in fields(metric):
                item[f.name] = getattr(metric, f.name)
            
            records.append(item)

    df = pd.DataFrame(records)
    return df

In [4]:
def sort_strategy(strategy_name: str) -> int:
    if "Greedy" in strategy_name or "greedy" in strategy_name.lower():
        return -1
    
    elif "_" in strategy_name:
        parts = strategy_name.split("_")
        if parts[1].isdigit():
            return int(parts[1])
    
    return 9999

def post_process_df(df: pd.DataFrame):
    grouped_df = df.groupby("Search Strategy").mean(numeric_only=True)
    sorted_strategies = sorted(df["Search Strategy"].unique(), key=sort_strategy)

    grouped_df = grouped_df.loc[sorted_strategies]
    grouped_df.reset_index(inplace=True)
    return grouped_df

def extract_and_format_metric(df: pd.DataFrame, values_col: str) -> pd.DataFrame:
    res_df = df.pivot(index="Search Strategy", columns="Species", values=values_col).T
    res_df.columns.name = None
    res_df = res_df.reset_index()
    sorted_strategies = sorted(res_df.columns[1:].unique(), key=sort_strategy)
    res_df = res_df[["Species", *sorted_strategies]]
    
    res_df.loc[len(res_df)] = ["mean", *res_df.mean(numeric_only=True).values]
    return res_df

# Realistic Data Augmentation

In [5]:
v1_df = calculate_multi_strategy_metrics(
    normalize_path("/data2/xp/RocNovo-Lightning/outputs/final/massiveKB_zero_shot_results"),
    "v1"
)
v2_df = calculate_multi_strategy_metrics(
    normalize_path("/data2/xp/RocNovo-Lightning/outputs/final/massiveKB_zero_shot_results"),
    "v2"
)
v1_grouped_df = post_process_df(v1_df)
v2_grouped_df = post_process_df(v2_df)

100%|██████████| 291783/291783 [00:12<00:00, 23683.10it/s]

100%|██████████| 36042/36042 [00:00<00:00, 36279.36it/s]s]]

100%|██████████| 1051672/1051672 [00:31<00:00, 33194.50it/s]


In [6]:
v1_grouped_df

,Search Strategy,aa_precision,aa_recall,peptide_recall,full_accuracy
0,greedy,0.808009,0.803457,0.628731,0.613238
1,beam_1,0.810217,0.804609,0.631807,0.615919
2,beam_2,0.817691,0.812590,0.643437,0.627042
3,beam_3,0.820553,0.815824,0.647415,0.630785
4,beam_4,0.822035,0.817505,0.649384,0.632668
5,beam_5,0.822828,0.818402,0.650515,0.633708
6,beam_6,0.823341,0.818989,0.651399,0.634560
7,beam_7,0.823789,0.819539,0.652070,0.635204
8,beam_8,0.824046,0.819848,0.652488,0.635596
9,beam_9,0.824269,0.820123,0.652738,0.635812


In [7]:
v2_grouped_df

,Search Strategy,aa_precision,aa_recall,peptide_recall,full_accuracy
0,greedy,0.917059,0.915275,0.791826,0.772180
1,beam_1,0.918562,0.916789,0.795582,0.775740
2,beam_2,0.921336,0.919578,0.801508,0.781456
3,beam_3,0.922243,0.920513,0.803139,0.783038
4,beam_4,0.922703,0.921005,0.803850,0.783691
5,beam_5,0.922983,0.921305,0.804222,0.784048
6,beam_6,0.923180,0.921508,0.804475,0.784299
7,beam_7,0.923304,0.921654,0.804673,0.784479
8,beam_8,0.923414,0.921774,0.804799,0.784604
9,beam_9,0.923578,0.921951,0.804923,0.784718


In [8]:
v1_peptide_recall_df = extract_and_format_metric(v1_df, "peptide_recall")
v1_aa_precision_df = extract_and_format_metric(v1_df, "aa_precision")
v1_aa_recall_df = extract_and_format_metric(v1_df, "aa_recall")
v1_full_accuracy = extract_and_format_metric(v1_df, "full_accuracy")

v2_peptide_recall_df = extract_and_format_metric(v2_df, "peptide_recall")
v2_aa_precision_df = extract_and_format_metric(v2_df, "aa_precision")
v2_aa_recall_df = extract_and_format_metric(v2_df, "aa_recall")
v2_full_accuracy = extract_and_format_metric(v2_df, "full_accuracy")

In [9]:
v1_peptide_recall_df

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.693903,0.692994,0.704907,0.709294,0.711320,0.712564,0.713469,0.714055,0.714380,...,0.715124,0.715350,0.715515,0.715631,0.715700,0.715813,0.715977,0.715926,0.715967,0.715998
1,clambacteria,0.499080,0.504512,0.513422,0.517087,0.518893,0.519776,0.520613,0.521157,0.521556,...,0.522485,0.522704,0.522777,0.522930,0.523063,0.523122,0.523030,0.523136,0.523235,0.523362
2,honeybee,0.587940,0.594047,0.606607,0.610784,0.613016,0.614322,0.615267,0.616115,0.616621,...,0.617336,0.617597,0.617778,0.617921,0.618013,0.618112,0.618217,0.618318,0.618328,0.618433
3,human,0.618741,0.625617,0.641699,0.647649,0.650919,0.652811,0.654274,0.655323,0.656142,...,0.657543,0.657888,0.658118,0.658325,0.658685,0.658845,0.659006,0.659236,0.659443,0.659558
4,mmazei,0.631501,0.628539,0.639462,0.642904,0.644705,0.645477,0.646231,0.646669,0.647040,...,0.647490,0.647600,0.647752,0.647831,0.647965,0.648141,0.648135,0.648226,0.648165,0.648178
5,mouse,0.612193,0.618244,0.630750,0.634532,0.636098,0.636936,0.637638,0.638340,0.638692,...,0.639286,0.639394,0.639772,0.640123,0.640366,0.640528,0.640123,0.640204,0.640366,0.640474
6,ricebean,0.690430,0.689583,0.703375,0.708167,0.710549,0.711794,0.713011,0.714017,0.714520,...,0.715632,0.716135,0.716188,0.716400,0.716453,0.716744,0.716929,0.717009,0.717088,0.717247
7,tomato,0.667840,0.672687,0.680983,0.683534,0.684875,0.685551,0.686064,0.686351,0.686571,...,0.686902,0.687047,0.687133,0.687137,0.687140,0.687282,0.687337,0.687378,0.687402,0.687461
8,yeast,0.656955,0.660037,0.669730,0.672785,0.674078,0.675408,0.676028,0.676603,0.676872,...,0.677241,0.677546,0.677645,0.677654,0.677708,0.677672,0.677968,0.678085,0.677941,0.678067
9,mean,0.628731,0.631807,0.643437,0.647415,0.649384,0.650515,0.651399,0.652070,0.652488,...,0.653227,0.653473,0.653631,0.653772,0.653899,0.654029,0.654080,0.654169,0.654215,0.654308


In [10]:
v1_aa_precision_df

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.848192,0.850690,0.859369,0.862822,0.864344,0.865376,0.865899,0.866498,0.866727,...,0.867355,0.867603,0.867792,0.867866,0.867913,0.868053,0.868192,0.868302,0.868394,0.868512
1,clambacteria,0.730615,0.732439,0.737178,0.738679,0.739337,0.739628,0.739816,0.739815,0.739831,...,0.740127,0.740096,0.740107,0.740160,0.740203,0.740243,0.740255,0.740236,0.740277,0.740318
2,honeybee,0.789851,0.793491,0.801749,0.804882,0.806562,0.807446,0.808138,0.808974,0.809369,...,0.809901,0.810222,0.810360,0.810569,0.810667,0.810792,0.810770,0.811005,0.811071,0.811157
3,human,0.789600,0.792754,0.803268,0.807374,0.809883,0.811307,0.812797,0.813738,0.814181,...,0.815539,0.815820,0.816128,0.816287,0.816672,0.816777,0.817135,0.817319,0.817448,0.817727
4,mmazei,0.813721,0.814793,0.821238,0.823791,0.825033,0.825624,0.826001,0.826381,0.826603,...,0.827226,0.827361,0.827473,0.827573,0.827580,0.827779,0.827764,0.827892,0.827906,0.827916
5,mouse,0.836472,0.837980,0.842624,0.844418,0.845394,0.845853,0.845885,0.845932,0.846398,...,0.846818,0.847048,0.847152,0.847255,0.847169,0.847358,0.847609,0.847179,0.847568,0.847551
6,ricebean,0.838116,0.840310,0.851769,0.857235,0.859805,0.861136,0.862027,0.862966,0.863584,...,0.864076,0.865034,0.864874,0.865215,0.865356,0.865407,0.865970,0.865826,0.866212,0.866341
7,tomato,0.827786,0.829241,0.833698,0.835036,0.835529,0.835864,0.836106,0.836165,0.836189,...,0.836282,0.836302,0.836301,0.836347,0.836324,0.836493,0.836562,0.836513,0.836568,0.836590
8,yeast,0.797728,0.800254,0.808326,0.810740,0.812421,0.813221,0.813396,0.813628,0.813534,...,0.813825,0.814026,0.814216,0.814157,0.814277,0.814089,0.814150,0.814253,0.814028,0.814349
9,mean,0.808009,0.810217,0.817691,0.820553,0.822035,0.822828,0.823341,0.823789,0.824046,...,0.824572,0.824835,0.824934,0.825048,0.825129,0.825221,0.825378,0.825392,0.825497,0.825607


In [11]:
v1_aa_recall_df

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.846096,0.846549,0.856571,0.860736,0.862621,0.863828,0.864476,0.865119,0.865375,...,0.866107,0.866375,0.866574,0.866643,0.866704,0.866851,0.866982,0.867099,0.867209,0.867322
1,clambacteria,0.722787,0.724492,0.729471,0.731140,0.732049,0.732466,0.732775,0.732878,0.733018,...,0.733468,0.733499,0.733575,0.733661,0.733709,0.733750,0.733809,0.733859,0.733884,0.734013
2,honeybee,0.789177,0.792494,0.800868,0.804150,0.805993,0.806979,0.807752,0.808596,0.809022,...,0.809690,0.810007,0.810217,0.810403,0.810527,0.810660,0.810657,0.810904,0.810963,0.811072
3,human,0.788381,0.790276,0.801106,0.805447,0.808039,0.809524,0.811091,0.812149,0.812653,...,0.814121,0.814440,0.814781,0.814960,0.815381,0.815454,0.815857,0.816048,0.816199,0.816511
4,mmazei,0.812353,0.810960,0.818867,0.822443,0.824087,0.824932,0.825414,0.825888,0.826162,...,0.826862,0.826990,0.827098,0.827247,0.827243,0.827440,0.827443,0.827567,0.827553,0.827614
5,mouse,0.832357,0.833864,0.838738,0.840680,0.841739,0.842242,0.842322,0.842422,0.842897,...,0.843322,0.843525,0.843635,0.843707,0.843671,0.843886,0.844166,0.843769,0.844158,0.844178
6,ricebean,0.824925,0.825114,0.837062,0.843064,0.845913,0.847394,0.848189,0.849372,0.849908,...,0.851073,0.851412,0.851685,0.852014,0.852035,0.852296,0.852523,0.852689,0.852898,0.853134
7,tomato,0.822020,0.823322,0.827962,0.829479,0.830069,0.830436,0.830770,0.830923,0.831045,...,0.831170,0.831221,0.831230,0.831377,0.831361,0.831486,0.831588,0.831543,0.831600,0.831609
8,yeast,0.793013,0.794411,0.802666,0.805275,0.807032,0.807817,0.808115,0.808504,0.808555,...,0.808969,0.809140,0.809305,0.809406,0.809525,0.809365,0.809486,0.809623,0.809548,0.809860
9,mean,0.803457,0.804609,0.812590,0.815824,0.817505,0.818402,0.818989,0.819539,0.819848,...,0.820531,0.820734,0.820900,0.821047,0.821128,0.821243,0.821390,0.821456,0.821557,0.821702


In [12]:
v1_full_accuracy

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.668850,0.667523,0.678689,0.682778,0.684588,0.685753,0.686589,0.687117,0.687442,...,0.688131,0.688344,0.688512,0.688577,0.688645,0.688752,0.688937,0.688844,0.688923,0.688940
1,clambacteria,0.481983,0.486950,0.495369,0.498782,0.500395,0.501099,0.501816,0.502307,0.502599,...,0.503350,0.503556,0.503682,0.503821,0.503941,0.503980,0.503894,0.503980,0.504060,0.504166
2,honeybee,0.580419,0.586246,0.598529,0.602598,0.604760,0.606006,0.606919,0.607736,0.608238,...,0.608925,0.609157,0.609351,0.609478,0.609573,0.609672,0.609770,0.609856,0.609872,0.609983
3,human,0.615647,0.622309,0.638169,0.643958,0.647198,0.649051,0.650544,0.651563,0.652344,...,0.653684,0.654059,0.654281,0.654496,0.654810,0.654993,0.655116,0.655369,0.655545,0.655652
4,mmazei,0.605957,0.602496,0.612829,0.615937,0.617598,0.618285,0.618966,0.619380,0.619708,...,0.620122,0.620201,0.620401,0.620456,0.620590,0.620705,0.620730,0.620778,0.620724,0.620760
5,mouse,0.601902,0.607466,0.619378,0.622998,0.624591,0.625402,0.626077,0.626725,0.627104,...,0.627671,0.627698,0.628049,0.628400,0.628616,0.628805,0.628400,0.628508,0.628616,0.628724
6,ricebean,0.671238,0.669993,0.683150,0.687598,0.690060,0.691066,0.692310,0.693342,0.693819,...,0.694772,0.695275,0.695354,0.695486,0.695539,0.695831,0.695989,0.696069,0.696148,0.696228
7,tomato,0.649395,0.653898,0.661741,0.664089,0.665327,0.665954,0.666395,0.666678,0.666864,...,0.667154,0.667330,0.667402,0.667385,0.667413,0.667533,0.667595,0.667609,0.667619,0.667675
8,yeast,0.643749,0.646390,0.655527,0.658330,0.659498,0.660755,0.661420,0.661986,0.662247,...,0.662489,0.662849,0.662911,0.662849,0.662947,0.662893,0.663190,0.663325,0.663181,0.663253
9,mean,0.613238,0.615919,0.627042,0.630785,0.632668,0.633708,0.634560,0.635204,0.635596,...,0.636255,0.636496,0.636660,0.636772,0.636897,0.637018,0.637069,0.637149,0.637188,0.637265


In [13]:
import pandas as pd

df_pep_rec = v1_peptide_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Peptide Recall"})
df_aa_pre  = v1_aa_precision_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Precision"})
df_aa_rec  = v1_aa_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Recall"})
df_full_acc = v1_full_accuracy[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Full Accuracy"})
df_merged = pd.concat([df_pep_rec, df_aa_pre, df_aa_rec, df_full_acc], axis=1).reset_index()
df_merged

,Species,Peptide Recall,AA Precision,AA Recall,Full Accuracy
0,bacillus,0.714942,0.867171,0.865899,0.687946
1,clambacteria,0.522273,0.740048,0.733324,0.503177
2,honeybee,0.617190,0.809855,0.809546,0.608785
3,human,0.657076,0.815002,0.813526,0.653247
4,mmazei,0.647442,0.827166,0.826758,0.620055
5,mouse,0.639151,0.846643,0.843184,0.627482
6,ricebean,0.715235,0.863967,0.850552,0.694454
7,tomato,0.686892,0.836353,0.831195,0.667185
8,yeast,0.677115,0.813851,0.808952,0.662381
9,mean,0.653035,0.824451,0.820326,0.636079


In [14]:
v2_peptide_recall_df

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.844877,0.848382,0.853413,0.854868,0.855521,0.855905,0.856121,0.856289,0.856425,...,0.856642,0.856700,0.856749,0.856784,0.856792,0.856811,0.856835,0.856856,0.856862,0.856876
1,clambacteria,0.616238,0.620575,0.625681,0.627459,0.628170,0.628653,0.629094,0.629478,0.629606,...,0.630075,0.630260,0.630260,0.630146,0.630374,0.630317,0.630246,0.630161,0.630132,0.630203
2,honeybee,0.753803,0.759455,0.767688,0.770351,0.771438,0.771982,0.772342,0.772702,0.772885,...,0.773320,0.773320,0.773408,0.773483,0.773496,0.773558,0.773537,0.773564,0.773687,0.773768
3,human,0.856945,0.860718,0.868293,0.870151,0.870928,0.871428,0.871733,0.871844,0.871955,...,0.872399,0.872538,0.872538,0.872843,0.872593,0.872760,0.872676,0.872621,0.872760,0.872815
4,mmazei,0.823754,0.827143,0.831731,0.833123,0.833674,0.834046,0.834234,0.834441,0.834602,...,0.834785,0.834877,0.834896,0.834946,0.834942,0.834983,0.835015,0.835010,0.835056,0.835107
5,mouse,0.687161,0.691933,0.702769,0.705154,0.706099,0.706496,0.706894,0.706844,0.707043,...,0.707540,0.707292,0.707490,0.707540,0.707490,0.707540,0.707640,0.707739,0.708087,0.707987
6,ricebean,0.847383,0.850813,0.855786,0.857030,0.857762,0.858052,0.858103,0.858383,0.858454,...,0.858584,0.858704,0.858684,0.858654,0.858694,0.858714,0.858764,0.858724,0.858795,0.858825
7,tomato,0.819312,0.821467,0.824571,0.825446,0.825972,0.826160,0.826349,0.826537,0.826618,...,0.826739,0.826746,0.826793,0.826860,0.826934,0.826847,0.826854,0.826968,0.826887,0.826854
8,yeast,0.876964,0.879747,0.883640,0.884665,0.885086,0.885272,0.885404,0.885540,0.885603,...,0.885722,0.885763,0.885775,0.885761,0.885799,0.885806,0.885866,0.885863,0.885830,0.885835
9,mean,0.791826,0.795582,0.801508,0.803139,0.803850,0.804222,0.804475,0.804673,0.804799,...,0.805090,0.805133,0.805177,0.805224,0.805235,0.805259,0.805270,0.805279,0.805344,0.805363


In [15]:
v2_aa_precision_df

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.930827,0.933046,0.936517,0.937711,0.938314,0.938679,0.938988,0.939172,0.939316,...,0.939624,0.939696,0.939779,0.939794,0.939858,0.939887,0.939966,0.940011,0.940041,0.940066
1,clambacteria,0.845584,0.846441,0.848539,0.848594,0.848802,0.849015,0.849172,0.849127,0.849076,...,0.849375,0.849359,0.849410,0.849389,0.849447,0.849415,0.849380,0.849340,0.849500,0.849541
2,honeybee,0.893429,0.896348,0.901140,0.902813,0.903623,0.904187,0.904421,0.904738,0.904911,...,0.905439,0.905510,0.905583,0.905686,0.905759,0.905839,0.905902,0.905897,0.906001,0.906078
3,human,0.947171,0.947873,0.950826,0.951864,0.952557,0.952666,0.952889,0.953180,0.953179,...,0.953326,0.953438,0.953552,0.953500,0.953574,0.953527,0.953460,0.953569,0.953756,0.953719
4,mmazei,0.930772,0.932477,0.935169,0.936094,0.936574,0.936848,0.937071,0.937239,0.937443,...,0.937694,0.937778,0.937826,0.937869,0.937907,0.937943,0.937918,0.937977,0.938001,0.938028
5,mouse,0.892837,0.893896,0.896316,0.897382,0.897629,0.898102,0.898281,0.898238,0.898534,...,0.898958,0.899035,0.899120,0.899072,0.899060,0.899116,0.899219,0.899328,0.899300,0.899198
6,ricebean,0.942286,0.944033,0.946614,0.947376,0.947808,0.947978,0.948180,0.948166,0.948295,...,0.948416,0.948445,0.948399,0.948547,0.948531,0.948571,0.948623,0.948618,0.948672,0.948615
7,tomato,0.926064,0.926856,0.928573,0.929118,0.929391,0.929530,0.929621,0.929776,0.929811,...,0.930006,0.930005,0.930045,0.930158,0.930149,0.930150,0.930136,0.930227,0.930178,0.930148
8,yeast,0.944565,0.946086,0.948327,0.949235,0.949629,0.949841,0.949995,0.950102,0.950161,...,0.950399,0.950431,0.950499,0.950554,0.950581,0.950636,0.950698,0.950657,0.950725,0.950717
9,mean,0.917059,0.918562,0.921336,0.922243,0.922703,0.922983,0.923180,0.923304,0.923414,...,0.923693,0.923744,0.923801,0.923841,0.923874,0.923898,0.923922,0.923958,0.924020,0.924012


In [16]:
v2_aa_recall_df

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.929295,0.931528,0.935005,0.936229,0.936853,0.937242,0.937564,0.937765,0.937921,...,0.938254,0.938332,0.938417,0.938440,0.938508,0.938539,0.938618,0.938670,0.938701,0.938726
1,clambacteria,0.841479,0.842428,0.844679,0.844859,0.845149,0.845369,0.845554,0.845567,0.845582,...,0.845965,0.845953,0.846026,0.846004,0.846068,0.846056,0.846030,0.845998,0.846156,0.846207
2,honeybee,0.891938,0.894883,0.899762,0.901455,0.902307,0.902902,0.903138,0.903488,0.903656,...,0.904244,0.904315,0.904374,0.904492,0.904577,0.904669,0.904708,0.904717,0.904812,0.904920
3,human,0.946332,0.947115,0.950128,0.951098,0.951846,0.952009,0.952183,0.952486,0.952495,...,0.952605,0.952735,0.952823,0.952797,0.952881,0.952850,0.952788,0.952885,0.953085,0.953035
4,mmazei,0.929900,0.931532,0.934153,0.935080,0.935557,0.935837,0.936053,0.936214,0.936416,...,0.936670,0.936746,0.936790,0.936834,0.936868,0.936898,0.936888,0.936947,0.936977,0.937005
5,mouse,0.889843,0.890827,0.893040,0.894152,0.894449,0.894911,0.895122,0.895060,0.895358,...,0.895786,0.895837,0.895935,0.895881,0.895859,0.895954,0.896066,0.896110,0.896128,0.896019
6,ricebean,0.940944,0.942689,0.945278,0.946063,0.946514,0.946718,0.946932,0.946953,0.947103,...,0.947249,0.947327,0.947290,0.947419,0.947433,0.947476,0.947530,0.947515,0.947579,0.947539
7,tomato,0.923725,0.924575,0.926372,0.926969,0.927246,0.927413,0.927517,0.927717,0.927738,...,0.927967,0.927987,0.928025,0.928122,0.928139,0.928125,0.928127,0.928211,0.928160,0.928148
8,yeast,0.944016,0.945527,0.947784,0.948709,0.949126,0.949346,0.949515,0.949635,0.949696,...,0.949941,0.949982,0.950049,0.950103,0.950130,0.950188,0.950242,0.950203,0.950273,0.950267
9,mean,0.915275,0.916789,0.919578,0.920513,0.921005,0.921305,0.921508,0.921654,0.921774,...,0.922076,0.922135,0.922192,0.922233,0.922274,0.922306,0.922333,0.922362,0.922430,0.922430


In [17]:
v2_full_accuracy

,Species,greedy,beam_1,beam_2,beam_3,beam_4,beam_5,beam_6,beam_7,beam_8,...,beam_11,beam_12,beam_13,beam_14,beam_15,beam_16,beam_17,beam_18,beam_19,beam_20
0,bacillus,0.818580,0.821819,0.826584,0.827924,0.828557,0.828887,0.829066,0.829222,0.829354,...,0.829549,0.829591,0.829641,0.829664,0.829672,0.829705,0.829711,0.829725,0.829739,0.829756
1,clambacteria,0.596698,0.600623,0.605287,0.607008,0.607662,0.608089,0.608530,0.608885,0.608985,...,0.609397,0.609554,0.609540,0.609497,0.609682,0.609639,0.609525,0.609497,0.609469,0.609540
2,honeybee,0.744414,0.749795,0.757763,0.760324,0.761384,0.761907,0.762226,0.762573,0.762736,...,0.763157,0.763116,0.763198,0.763279,0.763279,0.763347,0.763361,0.763436,0.763544,0.763592
3,human,0.853227,0.856945,0.864491,0.866295,0.867127,0.867599,0.867876,0.868015,0.868098,...,0.868570,0.868653,0.868681,0.868958,0.868736,0.868903,0.868792,0.868764,0.868875,0.868931
4,mmazei,0.785975,0.789098,0.793300,0.794609,0.795082,0.795440,0.795693,0.795789,0.795978,...,0.796194,0.796203,0.796203,0.796290,0.796285,0.796318,0.796396,0.796382,0.796373,0.796437
5,mouse,0.679308,0.684080,0.694716,0.697251,0.697997,0.698395,0.698792,0.698742,0.698991,...,0.699339,0.699289,0.699389,0.699538,0.699538,0.699538,0.699438,0.699637,0.700035,0.699836
6,ricebean,0.813932,0.817101,0.821944,0.823087,0.823699,0.823979,0.824030,0.824320,0.824381,...,0.824491,0.824581,0.824561,0.824461,0.824461,0.824511,0.824561,0.824631,0.824661,0.824701
7,tomato,0.797369,0.799429,0.802472,0.803334,0.803779,0.804021,0.804237,0.804398,0.804492,...,0.804560,0.804607,0.804614,0.804647,0.804769,0.804661,0.804701,0.804755,0.804748,0.804701
8,yeast,0.860118,0.862772,0.866548,0.867508,0.867929,0.868115,0.868240,0.868362,0.868422,...,0.868527,0.868565,0.868577,0.868544,0.868589,0.868603,0.868627,0.868632,0.868649,0.868651
9,mean,0.772180,0.775740,0.781456,0.783038,0.783691,0.784048,0.784299,0.784479,0.784604,...,0.784865,0.784907,0.784934,0.784986,0.785001,0.785025,0.785013,0.785051,0.785121,0.785127


In [18]:
import pandas as pd

df_pep_rec = v2_peptide_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Peptide Recall"})
df_aa_pre  = v2_aa_precision_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Precision"})
df_aa_rec  = v2_aa_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Recall"})
df_full_acc = v2_full_accuracy[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Full Accuracy"})
df_merged = pd.concat([df_pep_rec, df_aa_pre, df_aa_rec, df_full_acc], axis=1).reset_index()
df_merged

,Species,Peptide Recall,AA Precision,AA Recall,Full Accuracy
0,bacillus,0.856563,0.939550,0.938171,0.829479
1,clambacteria,0.629919,0.849310,0.845886,0.609269
2,honeybee,0.773259,0.905257,0.904042,0.763096
3,human,0.872232,0.953260,0.952554,0.868376
4,mmazei,0.834721,0.937699,0.936672,0.796116
5,mouse,0.707292,0.899004,0.895790,0.699240
6,ricebean,0.858554,0.948351,0.947193,0.824471
7,tomato,0.826719,0.929958,0.927916,0.804540
8,yeast,0.885706,0.950351,0.949891,0.868520
9,mean,0.804996,0.923638,0.922013,0.784789


# Dummy Data Augmentation

In [19]:
v1_df = calculate_multi_strategy_metrics(
    normalize_path("/data2/xp/RocNovo-Lightning/outputs/final/dummy_results"),
    "v1"
)
v2_df = calculate_multi_strategy_metrics(
    normalize_path("/data2/xp/RocNovo-Lightning/outputs/final/dummy_results"),
    "v2"
)
v1_grouped_df = post_process_df(v1_df)
v2_grouped_df = post_process_df(v2_df)

100%|██████████| 1051672/1051672 [00:30<00:00, 33950.78it/s]


In [20]:
v1_grouped_df

,Search Strategy,aa_precision,aa_recall,peptide_recall,full_accuracy
0,beam_10,0.825488,0.821792,0.653141,0.635959


In [21]:
v2_grouped_df

,Search Strategy,aa_precision,aa_recall,peptide_recall,full_accuracy
0,beam_10,0.923616,0.921955,0.804851,0.78449


In [22]:
v1_peptide_recall_df = extract_and_format_metric(v1_df, "peptide_recall")
v1_aa_precision_df = extract_and_format_metric(v1_df, "aa_precision")
v1_aa_recall_df = extract_and_format_metric(v1_df, "aa_recall")
v1_full_accuracy = extract_and_format_metric(v1_df, "full_accuracy")

v2_peptide_recall_df = extract_and_format_metric(v2_df, "peptide_recall")
v2_aa_precision_df = extract_and_format_metric(v2_df, "aa_precision")
v2_aa_recall_df = extract_and_format_metric(v2_df, "aa_recall")
v2_full_accuracy = extract_and_format_metric(v2_df, "full_accuracy")

In [23]:
import pandas as pd

df_pep_rec = v1_peptide_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Peptide Recall"})
df_aa_pre  = v1_aa_precision_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Precision"})
df_aa_rec  = v1_aa_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Recall"})
df_full_acc = v1_full_accuracy[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Full Accuracy"})
df_merged = pd.concat([df_pep_rec, df_aa_pre, df_aa_rec, df_full_acc], axis=1).reset_index()
df_merged

,Species,Peptide Recall,AA Precision,AA Recall,Full Accuracy
0,bacillus,0.716200,0.870201,0.869201,0.688611
1,clambacteria,0.518966,0.740473,0.733845,0.499877
2,honeybee,0.618843,0.812689,0.812601,0.609296
3,human,0.658600,0.816721,0.815483,0.654480
4,mmazei,0.649771,0.828166,0.827954,0.622712
5,mouse,0.635207,0.846246,0.842464,0.623835
6,ricebean,0.712429,0.861350,0.849743,0.690245
7,tomato,0.686464,0.835945,0.831248,0.666606
8,yeast,0.681786,0.817599,0.813585,0.667969
9,mean,0.653141,0.825488,0.821792,0.635959


In [24]:
import pandas as pd

df_pep_rec = v2_peptide_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Peptide Recall"})
df_aa_pre  = v2_aa_precision_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Precision"})
df_aa_rec  = v2_aa_recall_df[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "AA Recall"})
df_full_acc = v2_full_accuracy[["Species", "beam_10"]].set_index("Species").rename(columns={"beam_10": "Full Accuracy"})
df_merged = pd.concat([df_pep_rec, df_aa_pre, df_aa_rec, df_full_acc], axis=1).reset_index()
df_merged

,Species,Peptide Recall,AA Precision,AA Recall,Full Accuracy
0,bacillus,0.855605,0.939344,0.937926,0.828471
1,clambacteria,0.629919,0.848693,0.845017,0.605686
2,honeybee,0.774583,0.907340,0.906089,0.762940
3,human,0.871844,0.953019,0.952320,0.867654
4,mmazei,0.835479,0.937781,0.936820,0.797599
5,mouse,0.703415,0.896968,0.893752,0.695412
6,ricebean,0.858413,0.948250,0.947120,0.823498
7,tomato,0.826746,0.930073,0.927940,0.804075
8,yeast,0.887656,0.951074,0.950613,0.875079
9,mean,0.804851,0.923616,0.921955,0.784490
